# Exploratory Data Analysis

**Paper:** *Deep learning-based surrogate capacity models and multi-objective fragility estimates for reinforced concrete frames*
**Authors:** Lili Xing, Paolo Gardoni, Ge Song, Ying Zhou

## 1. Imports

In [2]:
import os
import numpy as np
import pandas as pd

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_DIR   = os.path.join(BASE_DIR, "1. Dataset", "DNN_dataset")
PROCESSED_DIR = os.path.join(BASE_DIR, "1. Dataset", "Python_DataProcess")

## 2. Inspect Raw Dataset Structure

In [3]:
# List the 50 simulation folders
folders = sorted(
    [d for d in os.listdir(DATASET_DIR) if d.startswith("RandomPushResults")],
    key=lambda x: int(x.replace("RandomPushResults", ""))
)
print(f"Folders: {len(folders)}  ({folders[0]} ... {folders[-1]})")

Folders: 80  (RandomPushResults1 ... RandomPushResults80)


In [4]:
# Files inside each folder
folder1 = os.path.join(DATASET_DIR, "RandomPushResults1")
for f in sorted(os.listdir(folder1)):
    path = os.path.join(folder1, f)
    if "PushOver_Random_S" not in f and os.path.isfile(path):
        with open(path) as fh:
            n = sum(1 for l in fh if l.strip())
        print(f"{f}  ->  {n} rows")

PushOver_Random_CapacityPointsR.txt  ->  2002 rows
PushOver_Random_InputVariablesR.txt  ->  2002 rows
PushOver_Random_T.txt  ->  2002 rows
PushOver_Random_availableS.txt  ->  2002 rows


In [5]:
# InputVariablesR: 23 input features per scenario
iv = np.loadtxt(os.path.join(folder1, "PushOver_Random_InputVariablesR.txt"))
print("InputVariables shape:", iv.shape)

InputVariables shape: (2002, 23)


In [6]:
# CapacityPointsR: 10 output columns per scenario
cp = np.loadtxt(os.path.join(folder1, "PushOver_Random_CapacityPointsR.txt"))
print("CapacityPoints shape:", cp.shape)
print()
print("Column mapping:")
for k, v in {
    0: "Dy  (mm)    -- top displacement at yield",
    1: "Vy  (kN)    -- base shear at yield  [top-disp curve]",
    2: "Du  (mm)    -- top displacement at peak",
    3: "Vu  (kN)    -- peak base shear  [= col 9]",
    4: "IDy_abs (mm)-- max inter-story drift at yield (absolute)",
    5: "IDu_abs (mm)-- max inter-story drift at peak  (absolute)",
    6: "IDy (frac)  -- max inter-story drift at yield (dimensionless)",
    7: "IVy (kN)    -- base shear at yield  [ISD curve]",
    8: "IDu (frac)  -- max inter-story drift at peak  (dimensionless)",
    9: "IVu (kN)    -- peak base shear  [ISD curve, = col 3]",
}.items():
    print(f"  col {k}: {v}")

CapacityPoints shape: (2002, 10)

Column mapping:
  col 0: Dy  (mm)    -- top displacement at yield
  col 1: Vy  (kN)    -- base shear at yield  [top-disp curve]
  col 2: Du  (mm)    -- top displacement at peak
  col 3: Vu  (kN)    -- peak base shear  [= col 9]
  col 4: IDy_abs (mm)-- max inter-story drift at yield (absolute)
  col 5: IDu_abs (mm)-- max inter-story drift at peak  (absolute)
  col 6: IDy (frac)  -- max inter-story drift at yield (dimensionless)
  col 7: IVy (kN)    -- base shear at yield  [ISD curve]
  col 8: IDu (frac)  -- max inter-story drift at peak  (dimensionless)
  col 9: IVu (kN)    -- peak base shear  [ISD curve, = col 3]


In [7]:
# Total scenarios across all 50 folders
total = 0
for folder in folders:
    with open(os.path.join(DATASET_DIR, folder, "PushOver_Random_availableS.txt")) as fh:
        total += sum(1 for l in fh if l.strip())
print(f"Total raw scenarios: {total:,}  ({total // len(folders)} per folder)")

Total raw scenarios: 160,160  (2002 per folder)


## 3. Column Names

### 23 Sampled Input Features (Table 1 of the paper)

| Col | Symbol | Description | Unit |
|-----|--------|-------------|------|
| 0 | B | Breadth of one span | m |
| 1 | D | Depth of one span | m |
| 2 | H | Story height | m |
| 3 | B_num | Number of spans (X) | — |
| 4 | D_num | Number of spans (Y) | — |
| 5 | H_num | Number of storeys | — |
| 6 | colWidth | Column width (square) | m |
| 7 | beamDepth | Beam depth | m |
| 8 | beamRat | Beam depth-to-width ratio | — |
| 9 | Asc | Steel bar area in column | mm² |
| 10 | Asb | Steel bar area in beam | mm² |
| 11 | t | Slab thickness | m |
| 12 | cover1 | Concrete cover – column | m |
| 13 | cover2 | Concrete cover – beam | m |
| 14 | Ec | Concrete elastic tangent | ×10¹⁰ Pa |
| 15 | nu_c | Concrete Poisson ratio | — |
| 16 | fc | Concrete compressive strength | ×10⁶ Pa |
| 17 | fcuRat | Ratio fcu / fc | — |
| 18 | eps_cu | Concrete ultimate strain (sampled offset from eps_c0) | — |
| 19 | Es | Steel elastic tangent | ×10¹⁰ Pa |
| 20 | nu_s | Steel Poisson ratio | — |
| 21 | fsy | Steel yield strength | ×10⁶ Pa |
| 22 | eta | Strain-hardening ratio | — |

### 6 Derived / Fixed Parameters (Table 1, dash rows)

| Symbol | Formula / Value | Description |
|--------|----------------|-------------|
| eps_c0 | `2 * fc / Ec` (unit-adjusted) | Concrete strain at peak compressive strength |
| lambda_ | `0.1` (fixed) | Concrete02 unloading slope ratio |
| fct | `0.1 * fc` | Concrete tensile strength (×10⁶ Pa) |
| Ecs | `fct / 0.002` | Concrete tension softening stiffness |
| R0, cR1, cR2 | `20, 0.9215, 0.15` (fixed) | SteelMPF transition parameters |
| a1, a2, a3, a4 | `0, 1, 0, 0` (fixed) | SteelMPF isotropic hardening parameters |

### 8 Target Variables

| Col | Symbol | Description | Unit |
|-----|--------|-------------|------|
| 23 | Dy | Top displacement at yield | m |
| 24 | Vy | Base shear at yield | kN |
| 25 | Du | Top displacement at peak | m |
| 26 | Vu | Peak base shear | kN |
| 27 | IDy | Max inter-story drift at yield | % |
| 28 | IVy | Base shear at yield (ISD curve) | kN |
| 29 | IDu | Max inter-story drift at peak | % |
| 30 | IVu | Peak base shear (ISD curve) = Vu | kN |

## 4. Load Pre-Processed Dataset

`Python_DataProcess/` contains the final compiled files (31 cols = 23 inputs + 8 outputs):
- `RandomPushover_DL_Total_Train_new80.txt` — 80 % split
- `RandomPushover_DL_Total_Test_new80.txt`  — 20 % split

In [8]:
INPUT_COLS = [
    "B", "D", "H", "B_num", "D_num", "H_num",
    "colWidth", "beamDepth", "beamRat",
    "Asc", "Asb", "t", "cover1", "cover2",
    "Ec", "nu_c", "fc", "fcuRat", "eps_cu",
    "Es", "nu_s", "fsy", "eta",
]
OUTPUT_COLS = ["Dy", "Vy", "Du", "Vu", "IDy", "IVy", "IDu", "IVu"]
ALL_COLS    = INPUT_COLS + OUTPUT_COLS

train_arr = np.loadtxt(os.path.join(PROCESSED_DIR, "RandomPushover_DL_Total_Train_new80.txt"))
test_arr  = np.loadtxt(os.path.join(PROCESSED_DIR, "RandomPushover_DL_Total_Test_new80.txt"))

print(f"Train : {train_arr.shape[0]:,} rows")
print(f"Test  : {test_arr.shape[0]:,} rows")
print(f"Total : {train_arr.shape[0] + test_arr.shape[0]:,} rows  x  {train_arr.shape[1]} cols")

Train : 105,116 rows
Test  : 5,533 rows
Total : 110,649 rows  x  31 cols


## 5. Build the Clean DataFrame

In [9]:
df_train = pd.DataFrame(train_arr, columns=ALL_COLS)
df_train["split"] = "train"

df_test = pd.DataFrame(test_arr, columns=ALL_COLS)
df_test["split"] = "test"

df = pd.concat([df_train, df_test], ignore_index=True)
print(f"Shape: {df.shape}")
df.head()

Shape: (110649, 32)


,B,D,H,B_num,D_num,H_num,colWidth,beamDepth,beamRat,Asc,...,eta,Dy,Vy,Du,Vu,IDy,IVy,IDu,IVu,split
0,3.872590,6.336304,4.928144,3.027105,3.212143,2.501537,0.591043,0.651065,0.378389,760.4216,...,0.016468,0.378479,2561.220,1.273374,2833.483,0.4928,2603.773,2.3984,2833.483,train
1,7.009406,3.241430,3.967510,2.281663,4.451455,5.059249,0.458534,0.740647,0.467227,281.3511,...,0.038714,0.203165,1467.264,0.538122,2325.176,0.2923,1448.344,0.8131,2325.176,train
2,8.587800,3.878700,4.854900,2.851900,4.516200,2.620700,1.028400,1.108600,0.460200,281.8100,...,0.036700,0.137770,5877.000,0.275300,8737.400,0.2200,6665.200,0.3700,8737.400,train
3,8.431800,6.559000,4.038600,2.268600,4.982600,1.623800,0.748700,1.092500,0.462200,368.0500,...,0.009300,0.134040,4767.100,0.244200,5891.100,0.2200,4767.100,0.4000,5891.100,train
4,4.032500,6.753100,3.339800,4.185100,3.183900,3.058300,1.106600,0.722300,0.372200,569.8600,...,0.039700,0.273880,6840.300,1.958100,10003.000,0.3400,6833.900,2.1200,10003.000,train


## 6. Add the 6 Derived / Fixed Parameters

These are not stored in the raw files but are required to fully define the OpenSees materials.
Computed directly from the sampled inputs using the formulas in Table 1.

In [10]:
# eps_c0 = 2 * fc / Ec  (dimensionless)
# fc is in x1e6 Pa, Ec is in x1e10 Pa  =>  eps_c0 = 2 * fc*1e6 / (Ec*1e10) = 2*fc / (Ec*1e4)
df["eps_c0"]  = 2.0 * df["fc"] / (df["Ec"] * 1e4)

df["lambda_"] = 0.1          # fixed: Concrete02 unloading slope ratio

df["fct"]     = 0.1 * df["fc"]           # tensile strength (x1e6 Pa)
df["Ecs"]     = df["fct"] / 0.002        # tension softening stiffness

df["R0"]      = 20.0          # fixed: SteelMPF transition parameter
df["cR1"]     = 0.9215        # fixed
df["cR2"]     = 0.15          # fixed

df["a1"]      = 0.0           # fixed: SteelMPF isotropic hardening
df["a2"]      = 1.0
df["a3"]      = 0.0
df["a4"]      = 0.0

DERIVED_COLS = ["eps_c0", "lambda_", "fct", "Ecs", "R0", "cR1", "cR2", "a1", "a2", "a3", "a4"]

print(f"Shape after derived columns: {df.shape}")
df[["fc", "Ec", "eps_c0", "fct", "Ecs"]].head()

Shape after derived columns: (110649, 43)


,fc,Ec,eps_c0,fct,Ecs
0,16.48022,3.194428,0.001032,1.648022,824.0110
1,25.06127,3.212880,0.001560,2.506127,1253.0635
2,28.49500,3.007600,0.001895,2.849500,1424.7500
3,25.65600,3.299100,0.001555,2.565600,1282.8000
4,25.08500,3.043900,0.001648,2.508500,1254.2500


## 7. Quick Checks

In [11]:
print("Missing values:", df.isnull().sum().sum())
print()

# Vu and IVu must be identical (peak base shear is the same on both pushover curves)
assert np.allclose(df["Vu"].values, df["IVu"].values)
print("Vu == IVu for all rows: True")
print()

# IDu is capped at 2.4% (paper's prescribed inter-story drift limit)
print(f"IDu max: {df['IDu'].max():.4f}%  (paper limit: 2.4%)")

Missing values: 0

Vu == IVu for all rows: True

IDu max: 2.3999%  (paper limit: 2.4%)


In [12]:
print("=== Sampled Inputs ===")
df[INPUT_COLS].describe().T.round(4)

=== Sampled Inputs ===


,count,mean,std,min,25%,50%,75%,max
B,110649.0,6.6375,1.8185,3.6000,5.0633,6.5605,8.1702,9.9996
D,110649.0,5.2840,1.5556,2.7000,3.9291,5.2215,6.6143,8.1000
H,110649.0,4.1198,0.6828,3.0000,3.5259,4.0823,4.6903,5.4000
B_num,110649.0,3.4788,0.8678,2.0000,2.7247,3.4661,4.2312,5.0000
D_num,110649.0,3.4754,0.8692,2.0000,2.7177,3.4607,4.2288,5.0000
H_num,110649.0,3.4905,1.5995,1.0000,2.1305,3.3256,4.7044,6.9992
colWidth,110649.0,0.7893,0.2478,0.3000,0.5865,0.8012,1.0026,1.2000
beamDepth,110649.0,0.8308,0.2234,0.4000,0.6477,0.8415,1.0231,1.2000
beamRat,110649.0,0.4155,0.0490,0.3300,0.3732,0.4157,0.4580,0.5000
Asc,110649.0,423.7656,213.2455,60.0170,239.0700,419.5873,607.6100,799.9922


In [13]:
print("=== Derived Parameters ===")
df[DERIVED_COLS].describe().T.round(6)

=== Derived Parameters ===


,count,mean,std,min,25%,50%,75%,max
eps_c0,110649.0,0.001347,0.000288,0.000779,0.001104,0.001348,0.001585,0.001996
lambda_,110649.0,0.100000,0.000000,0.100000,0.100000,0.100000,0.100000,0.100000
fct,110649.0,2.217763,0.459637,1.400000,1.824000,2.226000,2.616140,3.000000
Ecs,110649.0,1108.881332,229.818413,700.000000,912.000000,1113.000000,1308.070000,1500.000000
R0,110649.0,20.000000,0.000000,20.000000,20.000000,20.000000,20.000000,20.000000
cR1,110649.0,0.921500,0.000000,0.921500,0.921500,0.921500,0.921500,0.921500
cR2,110649.0,0.150000,0.000000,0.150000,0.150000,0.150000,0.150000,0.150000
a1,110649.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
a2,110649.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
a3,110649.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [14]:
print("=== Target Variables ===")
df[OUTPUT_COLS].describe().T.round(4)

=== Target Variables ===


,count,mean,std,min,25%,50%,75%,max
Dy,110649.0,0.2098,0.1099,0.0015,0.1301,0.1864,0.2660,0.9586
Vy,110649.0,3955.0448,2499.8420,3.5413,2135.0000,3416.5000,5191.9000,23817.8400
Du,110649.0,0.6591,0.4668,0.0247,0.3177,0.4759,0.8839,2.0003
Vu,110649.0,5146.9438,3110.7020,8.1935,2869.6700,4495.0000,6736.3000,31377.0000
IDy,110649.0,0.3132,0.1539,0.0024,0.2000,0.2805,0.3989,1.1000
IVy,110649.0,4063.5094,2575.6090,3.7656,2194.5900,3500.5000,5322.3130,24690.3700
IDu,110649.0,0.9727,0.6205,0.0400,0.4909,0.7387,1.3600,2.3999
IVu,110649.0,5146.9438,3110.7020,8.1935,2869.6700,4495.0000,6736.3000,31377.0000


## 8. Export to CSV

In [15]:
output_csv = os.path.join(BASE_DIR, "1. Dataset", "DNN_dataset_cleaned.csv")
df.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")
print(f"Shape: {df.shape}  ({os.path.getsize(output_csv)/1024**2:.1f} MB)")

Saved: c:\Users\DELL\Desktop\Prof. Gardoni Paper Reproduction Work\1. Dataset\DNN_dataset_cleaned.csv
Shape: (110649, 43)  (35.1 MB)


In [16]:
# Verify reload
df_check = pd.read_csv(output_csv)
print(f"Reloaded: {df_check.shape}")
print(f"Columns : {list(df_check.columns)}") 

Reloaded: (110649, 43)
Columns : ['B', 'D', 'H', 'B_num', 'D_num', 'H_num', 'colWidth', 'beamDepth', 'beamRat', 'Asc', 'Asb', 't', 'cover1', 'cover2', 'Ec', 'nu_c', 'fc', 'fcuRat', 'eps_cu', 'Es', 'nu_s', 'fsy', 'eta', 'Dy', 'Vy', 'Du', 'Vu', 'IDy', 'IVy', 'IDu', 'IVu', 'split', 'eps_c0', 'lambda_', 'fct', 'Ecs', 'R0', 'cR1', 'cR2', 'a1', 'a2', 'a3', 'a4']
